# 07. Unión de NASA POWER al panel y dataset final

**Propósito.** Calcular variables derivadas del clima **diario** (días sin lluvia, precipitación acumulada de 30 y 120 días), agregarlas a nivel semanal junto con el clima de NASA POWER, y unirlas al panel con SISAM. El resultado es el dataset final del preprocesamiento (entrada del EDA).

**Entradas:** `datos/datasets_preprocesamiento/panel_semanal_sisam.csv` y `NasaPower/nasa_power_limpio.csv`.

**Salida:** `datos/datasets_eda/dataset_panel_semanal_COMPLETO.csv`. No se sobrescriben los `*_COMPLETO.csv` del Hito 2 (quedan intactos junto a este).

**Qué cambió respecto al Hito 2**
- Solo la parte semanal (ya no hay unión por alerta ni ventanas de 7 días por alerta).
- Los derivados se calculan sobre la serie **diaria** antes de agregar y se toman al cierre de la semana (domingo).
- Variables nuevas: `T2M_MAX`, `precip_nasa_sem`, `dias_sin_lluvia_nasa`, `precip_acum_30d`, `precip_acum_120d`.

In [1]:
from pathlib import Path
import unicodedata

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 1. Entorno: Google Colab (Drive) o ejecucion local
# ---------------------------------------------------------------------------
USAR_DRIVE = True   # True en Colab; False para trabajar con la carpeta local del repositorio
BASE_DRIVE = Path("/content/drive/MyDrive/BIODIVERSITY_GUARD")

if USAR_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = BASE_DRIVE
    except ImportError:
        print("No se detecto Google Colab: se usa la raiz local del repositorio.")
        USAR_DRIVE = False

if not USAR_DRIVE:
    # la raiz es la primera carpeta (hacia arriba desde la actual) que contiene "datos"
    _aqui = Path.cwd().resolve()
    BASE = next((p for p in [_aqui, *_aqui.parents] if (p / "datos").is_dir()), None)
    if BASE is None:
        raise FileNotFoundError("No se encontro la carpeta 'datos' subiendo desde " + str(_aqui))

if not BASE.exists():
    raise FileNotFoundError(f"No existe BASE = {BASE}. Verifica la carpeta BIODIVERSITY_GUARD en Drive.")

# ---------------------------------------------------------------------------
# 2. Rutas (todas salen de BASE)
# ---------------------------------------------------------------------------
DIR_PREP = BASE / "datos" / "datasets_preprocesamiento"
DIR_BDQ = DIR_PREP / "BDQueimadas"
DIR_DETER = DIR_PREP / "DETER"
DIR_NASA = DIR_PREP / "NasaPower"
DIR_SISAM = DIR_PREP / "SISAM"
DIR_EDA = BASE / "datos" / "datasets_eda"

# ---------------------------------------------------------------------------
# 3. Constantes comunes (identicas en los 7 notebooks)
# ---------------------------------------------------------------------------
FECHA_INICIO_DATOS = "2022-01-01"
FECHA_FIN_EXTRACCION = "2026-11-01"      # DETER y BDQueimadas
PRIMERA_SEMANA = "2022-01-03"             # primer lunes con semana completa (2022-01-01 es sabado)
ULTIMA_SEMANA = "2026-10-19"              # lunes de la ultima semana completa (termina domingo 2026-10-25)
SATELITE_REFERENCIA = "AQUA_M-T"
ESTADOS_AMAZONIA_LEGAL = ["ACRE", "AMAPÁ", "AMAZONAS", "MARANHÃO", "MATO GROSSO", "PARÁ", "RONDÔNIA", "RORAIMA", "TOCANTINS"]
MAPA_UF = {"ACRE": "AC", "AMAPÁ": "AP", "AMAZONAS": "AM", "MARANHÃO": "MA", "MATO GROSSO": "MT", "PARÁ": "PA", "RONDÔNIA": "RO", "RORAIMA": "RR", "TOCANTINS": "TO"}
CORRECCIONES_MUNICIPIO = {"ELDORADO DOS CARAJAS": "ELDORADO DO CARAJAS", "SANTA ISABEL DO PARA": "SANTA IZABEL DO PARA"}
FECHA_CORTE_CENTROIDES = "2024-12-31"     # los centroides se fijan con focos hasta esta fecha

# ---------------------------------------------------------------------------
# 4. Funciones comunes
# ---------------------------------------------------------------------------
def normalizar_texto(texto):
    # mayusculas, sin tildes ni espacios sobrantes
    if pd.isna(texto):
        return texto
    texto = str(texto).upper().strip()
    return unicodedata.normalize("NFKD", texto).encode("ASCII", "ignore").decode("utf-8")


def semana_lunes(fecha):
    # lunes de la semana (lunes a domingo) de cada fecha; recibe una Serie de fechas
    f = pd.to_datetime(fecha).dt.normalize()
    return f - pd.to_timedelta(f.dt.day_of_week, unit="D")


SEMANAS = pd.date_range(PRIMERA_SEMANA, ULTIMA_SEMANA, freq="W-MON")   # calendario de semanas completas
assert len(SEMANAS) == 251, f"El calendario deberia tener 251 semanas y tiene {len(SEMANAS)}"
assert (SEMANAS.day_of_week == 0).all(), "Todas las semanas deben empezar en lunes"


def verificar_entradas(archivos):
    # imprime los archivos de entrada con su tamano y falla si falta alguno
    faltan = []
    for a in map(Path, archivos):
        if a.exists():
            print(f"  OK  {a.relative_to(BASE)}  ({a.stat().st_size / 1e6:,.1f} MB)")
        else:
            faltan.append(a)
    if faltan:
        lista = "\n".join(f"  - {a}" for a in faltan)
        raise FileNotFoundError("Faltan archivos de entrada (copialos a esas carpetas y vuelve a ejecutar):\n" + lista)


print("BASE:", BASE)
print("Calendario:", SEMANAS[0].date(), "a", SEMANAS[-1].date(), f"({len(SEMANAS)} semanas)")

RUTA_PANEL = DIR_PREP / "panel_semanal_sisam.csv"
RUTA_NASA = DIR_NASA / "nasa_power_limpio.csv"
RUTA_SALIDA = DIR_EDA / "dataset_panel_semanal_COMPLETO.csv"
CLAVE = ["municipio_norm", "uf"]
VARIABLES_NASA = ["T2M", "RH2M", "WS2M", "PRECTOTCORR", "T2M_MAX"]

# Convencion: un dia es "seco" si llueve menos de este umbral (mm). Parametro editable.
UMBRAL_DIA_SECO_MM = 1.0
VENTANAS_PRECIP_DIAS = (30, 120)

COLUMNAS_FINALES = [
    "semana", "municipio_norm", "uf", "lat", "lon",
    "num_incendios", "frp_total", "frp_promedio", "precipitacion_promedio", "dias_sin_lluvia_promedio",
    "riesgo_fuego_promedio", "riesgo_fuego_max",
    "deter_num_alertas", "deter_area_ha", "deter_num_alertas_area_protegida",
    "pm10_reanalise", "pm2_5_reanalise", "o3_reanalise", "no2_reanalise", "co_reanalise", "so2_reanalise",
    "dias_sisam", "fuente_sisam",
    "T2M", "RH2M", "WS2M", "T2M_MAX", "precip_nasa_sem", "dias_sin_lluvia_nasa", "precip_acum_30d", "precip_acum_120d",
]

DIR_EDA.mkdir(parents=True, exist_ok=True)
print("\nArchivos de entrada:")
verificar_entradas([RUTA_PANEL, RUTA_NASA])

Mounted at /content/drive
BASE: /content/drive/MyDrive/BIODIVERSITY_GUARD
Calendario: 2022-01-03 a 2026-10-19 (251 semanas)

Archivos de entrada:
  OK  datos/datasets_preprocesamiento/panel_semanal_sisam.csv  (29.0 MB)
  OK  datos/datasets_preprocesamiento/NasaPower/nasa_power_limpio.csv  (55.9 MB)


In [2]:
panel = pd.read_csv(RUTA_PANEL, parse_dates=["semana"])
nasa = pd.read_csv(RUTA_NASA, parse_dates=["fecha"])
filas_panel = len(panel)

faltan = [v for v in VARIABLES_NASA if v not in nasa.columns]
assert not faltan, f"nasa_power_limpio.csv no tiene las variables {faltan}: revisar el notebook 06"
print("Panel con SISAM:", panel.shape)
print("NASA POWER diario:", nasa.shape, "| rango:", nasa["fecha"].min().date(), "a", nasa["fecha"].max().date())
assert not nasa.duplicated(["fecha"] + CLAVE).any(), "NASA POWER tiene filas duplicadas (fecha, municipio)"

pares_panel = set(map(tuple, panel[CLAVE].drop_duplicates().values))
pares_nasa = set(map(tuple, nasa[CLAVE].drop_duplicates().values))
assert pares_panel <= pares_nasa, f"Municipios del panel sin NASA POWER: {sorted(pares_panel - pares_nasa)[:10]}"

Panel con SISAM: (136293, 23)
NASA POWER diario: (1021383, 8) | rango: 2021-09-01 a 2026-10-25


## 1. Derivados sobre la serie diaria

Se calculan **antes** de agregar a semana, con la serie diaria ordenada por municipio y fecha:

- `dias_sin_lluvia_nasa`: días consecutivos, hasta ese día, con `PRECTOTCORR` menor que `UMBRAL_DIA_SECO_MM`. Si ese día no hay dato de precipitación, el valor es NaN y la cuenta se reinicia (criterio conservador).
- `precip_acum_30d` y `precip_acum_120d`: suma móvil de `PRECTOTCORR` con **ventana completa**: NaN si falta algún día de la ventana. Por eso NASA POWER se descargó desde 2021-09-01: la ventana de 120 días ya está completa en la primera semana del panel.

Antes de calcular, cada municipio se reindexa a todos los días del periodo (si faltara algún día, queda como NaN y no se salta).

In [3]:
def calcular_derivados_diarios(nasa_diario, umbral=UMBRAL_DIA_SECO_MM, ventanas=VENTANAS_PRECIP_DIAS):
    # devuelve la serie diaria completa con dias_sin_lluvia_nasa y precip_acum_{n}d
    dias = pd.date_range(nasa_diario["fecha"].min(), nasa_diario["fecha"].max(), freq="D")
    pares = nasa_diario[CLAVE].drop_duplicates().sort_values(CLAVE)
    d = pares.merge(pd.DataFrame({"fecha": dias}), how="cross").merge(nasa_diario, on=CLAVE + ["fecha"], how="left")
    d = d.sort_values(CLAVE + ["fecha"]).reset_index(drop=True)
    d["_id"] = d.groupby(CLAVE, sort=False).ngroup()

    p = d["PRECTOTCORR"]
    seco = p < umbral                                       # NaN cuenta como no seco
    bloque = (~seco).groupby(d["_id"]).cumsum()             # sube en cada dia no seco
    racha = seco.astype(int).groupby([d["_id"], bloque]).cumsum()
    d["dias_sin_lluvia_nasa"] = racha.where(p.notna())

    for n in ventanas:
        d[f"precip_acum_{n}d"] = d.groupby("_id")["PRECTOTCORR"].transform(lambda s: s.rolling(n, min_periods=n).sum())
    return d.drop(columns="_id")


# prueba minima de la logica (serie sintetica): 3 dias secos, uno con lluvia, 2 secos, un dato faltante
_t = pd.DataFrame({"fecha": pd.date_range("2022-01-01", periods=7), "municipio_norm": "X", "uf": "AA",
                   "T2M": 25.0, "RH2M": 80.0, "WS2M": 1.0, "T2M_MAX": 30.0,
                   "PRECTOTCORR": [0.0, 0.5, 0.2, 5.0, 0.0, 0.9, np.nan]})
_r = calcular_derivados_diarios(_t)["dias_sin_lluvia_nasa"].tolist()
assert _r[:6] == [1, 2, 3, 0, 1, 2] and np.isnan(_r[6]), _r
print("Prueba de dias_sin_lluvia_nasa OK:", _r)

diario = calcular_derivados_diarios(nasa)
print("Serie diaria con derivados:", diario.shape)
print(diario[["dias_sin_lluvia_nasa", "precip_acum_30d", "precip_acum_120d"]].describe())

Prueba de dias_sin_lluvia_nasa OK: [1.0, 2.0, 3.0, 0.0, 1.0, 2.0, nan]
Serie diaria con derivados: (1021383, 11)
       dias_sin_lluvia_nasa  precip_acum_30d  precip_acum_120d
count          1.021383e+06     1.005636e+06     956766.000000
mean           5.332204e+00     1.313546e+02        534.687928
std            1.346947e+01     1.152467e+02        373.992633
min            0.000000e+00     0.000000e+00          0.120000
25%            0.000000e+00     3.181000e+01        210.480000
50%            0.000000e+00     1.064600e+02        488.420000
75%            4.000000e+00     2.026900e+02        782.090000
max            1.720000e+02     9.188600e+02       1987.440000


### Hallazgos de esta etapa

**Qué se observa**
- La prueba de `dias_sin_lluvia_nasa` pasó: `[1.0, 2.0, 3.0, 0.0, 1.0, 2.0, nan]`.
- La serie diaria tiene **1,021,383** filas. Los acumulados tienen nulos solo al inicio de cada serie: `precip_acum_30d` con **1,005,636** valores y `precip_acum_120d` con **956,766**.
- Medias: `dias_sin_lluvia_nasa` **5.33** días (máximo **172**), `precip_acum_30d` **131.35** mm y `precip_acum_120d` **534.69** mm.

**Por qué se decidió así**
- `dias_sin_lluvia_nasa` cuenta los días seguidos con `PRECTOTCORR` menor que **1.0 mm** (`UMBRAL_DIA_SECO_MM`). Si falta el dato de un día, la cuenta se reinicia.
- `precip_acum_30d` y `precip_acum_120d` son sumas móviles con **ventana completa**. Si falta un día de la ventana, el valor es nulo.
- Se calculan sobre la serie diaria y se toman el domingo, para que describan la semana ya cerrada.

**Qué implica para el modelado**
- Estas variables resumen la sequía acumulada y son candidatas para explicar el riesgo de fuego.


## 2. Agregación semanal

Por `(semana, municipio_norm, uf)`:
- Promedio semanal de `T2M`, `RH2M` y `WS2M`; máximo semanal de `T2M_MAX`.
- `precip_nasa_sem`: suma semanal de `PRECTOTCORR`. **Decisión:** si la semana tiene algún día sin dato, la suma queda en NaN (una suma parcial subestimaría la lluvia).
- Los tres derivados (`dias_sin_lluvia_nasa`, `precip_acum_30d`, `precip_acum_120d`) se toman **al cierre de la semana (domingo)**.

Solo entran las 251 semanas del calendario; los días anteriores (la holgura desde 2021-09-01) ya cumplieron su función en las ventanas móviles.

In [4]:
diario["semana"] = semana_lunes(diario["fecha"])
dentro = diario[diario["semana"].isin(SEMANAS)]
print(f"Dias diarios fuera del calendario (holgura inicial y posteriores): {len(diario) - len(dentro):,}")

semanal = (
    dentro.groupby(["semana"] + CLAVE)
    .agg(T2M=("T2M", "mean"), RH2M=("RH2M", "mean"), WS2M=("WS2M", "mean"), T2M_MAX=("T2M_MAX", "max"),
         precip_nasa_sem=("PRECTOTCORR", lambda s: s.sum(min_count=7)))
    .reset_index()
)

domingo = dentro[dentro["fecha"] == dentro["semana"] + pd.Timedelta(days=6)]
cierre = domingo[["semana"] + CLAVE + ["dias_sin_lluvia_nasa", "precip_acum_30d", "precip_acum_120d"]]
semanal = semanal.merge(cierre, on=["semana"] + CLAVE, how="left", validate="one_to_one")
print("NASA POWER semanal:", semanal.shape)

Dias diarios fuera del calendario (holgura inicial y posteriores): 67,332
NASA POWER semanal: (136293, 11)


In [5]:
final = panel.merge(semanal, on=["semana"] + CLAVE, how="left", validate="one_to_one")

assert len(final) == filas_panel, f"El merge cambio el numero de filas: {filas_panel} -> {len(final)}"
assert not final.duplicated(CLAVE + ["semana"]).any(), "Clave duplicada tras el merge"
assert list(final.columns) == COLUMNAS_FINALES or set(final.columns) == set(COLUMNAS_FINALES), \
    f"Columnas distintas de las esperadas: {set(final.columns) ^ set(COLUMNAS_FINALES)}"
final = final[COLUMNAS_FINALES].sort_values(CLAVE + ["semana"]).reset_index(drop=True)
print("Dataset final:", final.shape)

Dataset final: (136293, 31)


## 3. Controles finales

Rango y número de semanas, número de municipios, filas = municipios por semanas, nulos por columna y por año, y confirmación de que no hay semanas fuera del calendario.

In [6]:
n_mun = final[CLAVE].drop_duplicates().shape[0]
assert final["semana"].min() == pd.Timestamp(PRIMERA_SEMANA) and final["semana"].max() == pd.Timestamp(ULTIMA_SEMANA)
assert final["semana"].nunique() == len(SEMANAS) == 251
assert set(final["semana"].unique()) == set(SEMANAS), "Hay semanas fuera del calendario o faltan semanas"
assert len(final) == n_mun * 251, "Filas != municipios x semanas"
assert not final.duplicated(CLAVE + ["semana"]).any()
assert list(final.columns) == COLUMNAS_FINALES

print(f"Semanas: {final['semana'].nunique()} ({final['semana'].min().date()} a {final['semana'].max().date()})")
print(f"Municipios: {n_mun} | filas: {len(final):,} = {n_mun} x 251")
print("\nNulos por columna:")
print(final.isna().sum()[lambda s: s > 0])
print("\nNulos por anio (conteo):")
print(final.groupby(final["semana"].dt.year)[COLUMNAS_FINALES[3:]].apply(lambda g: g.isna().sum()).T)

Semanas: 251 (2022-01-03 a 2026-10-19)
Municipios: 543 | filas: 136,293 = 543 x 251

Nulos por columna:
frp_promedio                97131
precipitacion_promedio      97600
dias_sin_lluvia_promedio    98813
riesgo_fuego_promedio       97977
riesgo_fuego_max            97977
dtype: int64

Nulos por anio (conteo):
semana                             2022   2023   2024   2025   2026
lat                                   0      0      0      0      0
lon                                   0      0      0      0      0
num_incendios                         0      0      0      0      0
frp_total                             0      0      0      0      0
frp_promedio                      19918  18952  19191  21262  17808
precipitacion_promedio            19918  18952  19191  21528  18011
dias_sin_lluvia_promedio          20146  19237  19518  21763  18149
riesgo_fuego_promedio             20011  19057  19287  21587  18035
riesgo_fuego_max                  20011  19057  19287  21587  18035
deter_n

### Hallazgos de esta etapa

**Qué se observa**
- El dataset final tiene **136,293 filas x 31 columnas** (543 municipios x 251 semanas).
- Las columnas de NASA POWER, SISAM, DETER, `lat` y `lon` no tienen nulos en ningún año.
- Los nulos de `frp_promedio` son **97,131** en total (**19,918**, **18,952**, **19,191**, **21,262** y **17,808** de 2022 a 2026). Son las semanas sin focos.
- Las demás columnas de focos tienen más nulos que `frp_promedio`: `precipitacion_promedio` **97,600**, `dias_sin_lluvia_promedio` **98,813** y `riesgo_fuego_promedio` y `riesgo_fuego_max` **97,977** cada una.

**Lectura del desplazamiento entre años.** Los focos de 2025 (**43,131**) son mucho menos que los de 2024 (**140,346**), y el porcentaje de semanas con focos baja de **33.3 %** a **24.7 %** (2026: **21.9 %**). Los nulos de `riesgo_fuego_promedio` por encima de los de `frp_promedio` son **93**, **105** y **96** en 2022 a 2024, y suben a **325** en 2025 y **227** en 2026. El PM2.5 de 2026 viene de pronóstico (media **7.58** frente a **16.10** del reanálisis). Las causas no se pueden confirmar desde el preprocesamiento.

**Por qué se decidió así**
- Los nulos de focos no se imputan en el preprocesamiento. Se dejan para la preparación de datos, donde se podrá decidir con un criterio único.

**Qué implica para el modelado**
- **Limitación:** entrenar con 2022 a 2024 y probar con 2025 y 2026 enfrenta al modelo a un nivel de focos, de nulos de `RiscoFogo` y de PM2.5 distinto al que vio.


In [7]:
final.to_csv(RUTA_SALIDA, index=False, encoding="utf-8", date_format="%Y-%m-%d")
print("Guardado:", RUTA_SALIDA.relative_to(BASE), final.shape)
final.head()

Guardado: datos/datasets_eda/dataset_panel_semanal_COMPLETO.csv (136293, 31)


,semana,municipio_norm,uf,lat,lon,num_incendios,frp_total,frp_promedio,precipitacion_promedio,dias_sin_lluvia_promedio,...,dias_sisam,fuente_sisam,T2M,RH2M,WS2M,T2M_MAX,precip_nasa_sem,dias_sin_lluvia_nasa,precip_acum_30d,precip_acum_120d
0,2022-01-03,ABAETETUBA,PA,-1.773752,-48.785494,0,0.0,NaN,NaN,NaN,...,7,reanalise,25.572857,91.492857,0.095714,29.58,60.85,0,207.76,788.17
1,2022-01-10,ABAETETUBA,PA,-1.773752,-48.785494,0,0.0,NaN,NaN,NaN,...,7,reanalise,25.284286,91.577143,0.125714,29.42,78.69,0,233.61,818.49
2,2022-01-17,ABAETETUBA,PA,-1.773752,-48.785494,0,0.0,NaN,NaN,NaN,...,7,reanalise,25.952857,91.734286,0.095714,29.92,60.48,0,282.75,828.26
3,2022-01-24,ABAETETUBA,PA,-1.773752,-48.785494,0,0.0,NaN,NaN,NaN,...,7,reanalise,24.961429,92.440000,0.112857,29.55,73.82,0,287.51,828.38
4,2022-01-31,ABAETETUBA,PA,-1.773752,-48.785494,0,0.0,NaN,NaN,NaN,...,7,reanalise,25.761429,93.367143,0.092857,29.28,74.52,0,311.23,875.14


## Diccionario de variables del dataset final

Archivo `dataset_panel_semanal_COMPLETO.csv`. Una fila por municipio y semana (la semana se identifica por su lunes; va de lunes a domingo).

| Variable | Descripción | Fuente |
|---|---|---|
| `semana` | Lunes de la semana. 251 semanas, de 2022-01-03 a 2026-10-19. | Calendario |
| `municipio_norm`, `uf` | Municipio (mayúsculas, sin tildes) y estado (sigla). Clave junto con `semana`. | BDQueimadas y DETER |
| `lat`, `lon` | Centroide fijo del municipio (promedio de focos de 2022 a 2024). | BDQueimadas |
| `num_incendios` | Número de focos de calor de la semana (0 si no hubo). | BDQueimadas |
| `frp_total`, `frp_promedio` | Potencia radiativa del fuego (FRP, MW): suma y promedio de los focos de la semana. El promedio es NaN sin focos. | BDQueimadas |
| `precipitacion_promedio` | Precipitación promedio reportada en los focos de la semana. NaN sin focos. | BDQueimadas |
| `dias_sin_lluvia_promedio` | Promedio de días sin lluvia reportado en los focos. NaN sin focos. | BDQueimadas |
| `riesgo_fuego_promedio`, `riesgo_fuego_max` | Promedio y máximo de `RiscoFogo` (0 a 1) de los focos de la semana. NaN sin focos o sin dato. | BDQueimadas |
| `deter_num_alertas` | Alertas DETER de la semana (solo contexto para el KPI). | DETER |
| `deter_area_ha` | Hectáreas de las alertas DETER de la semana (solo contexto). | DETER |
| `deter_num_alertas_area_protegida` | Alertas DETER dentro de unidades de conservación (solo contexto). | DETER |
| `pm10_reanalise`, `pm2_5_reanalise`, `o3_reanalise`, `no2_reanalise`, `co_reanalise`, `so2_reanalise` | Promedio semanal de contaminantes. NaN si la semana tiene menos de 7 días de dato. Para 2026 provienen del pronóstico (mismas columnas). | SISAM |
| `dias_sisam` | Días con dato de SISAM en la semana (0 a 7). | SISAM |
| `fuente_sisam` | `reanalise`, `pronostico` o `mixta` (la semana mezcla ambas). | SISAM |
| `T2M`, `RH2M`, `WS2M` | Promedio semanal de temperatura a 2 m (°C), humedad relativa a 2 m (%) y viento a 2 m (m/s). | NASA POWER |
| `T2M_MAX` | Temperatura máxima semanal a 2 m (°C). | NASA POWER |
| `precip_nasa_sem` | Precipitación total de la semana (mm, `PRECTOTCORR`). NaN si falta algún día. | NASA POWER |
| `dias_sin_lluvia_nasa` | Días consecutivos con lluvia menor que `UMBRAL_DIA_SECO_MM` (1.0 mm), al domingo. | NASA POWER (derivada) |
| `precip_acum_30d`, `precip_acum_120d` | Precipitación acumulada de los 30 y 120 días hasta el domingo (ventana completa). | NASA POWER (derivada) |

Este dataset es crudo e integrado: no incluye rezagos, targets, recorte de atípicos, imputación ni transformaciones (eso es Data Preparation).

### Resumen para el informe

- **Esqueleto del dataset:** BDQueimadas (focos de calor, satélite `AQUA_M-T`) es la base. El panel completo tiene **543 municipios x 251 semanas = 136,293 filas** y **31 columnas**, con ceros reales en las semanas sin focos. La semana va de lunes a domingo.
- **DETER solo como contexto:** la extracción nueva trae **174,389** alertas para 2022 a 2024 contra **49,971** del Hito 2. **Limitación:** el extracto del Hito 2 estaba truncado por el tope de 50,000 filas del servicio.
- **Cobertura:** **Limitación:** el panel incluye solo los municipios con focos del bioma Amazônia. Se excluyeron **9** municipios sin focos y **242** alertas DETER.
- **Desplazamiento de focos:** **43,131** focos en 2025 contra **140,346** en 2024. El porcentaje de semanas con focos baja a **24.7 %** (2025) y **21.9 %** (2026).
- **Nulos de `RiscoFogo`:** pasan de menos de **0.5 %** en 2022 a 2024 a **6.74 %** en 2025 y **10.18 %** en 2026. **Limitación:** afecta el target de clasificación y no se imputó en el preprocesamiento.
- **Calidad del aire:** 2022 a 2025 usa reanálisis y 2026 usa pronóstico sin calibrar (PM2.5: **16.10** frente a **7.58**). **Limitación:** el sufijo `_forecast_0` se interpretó como plazo 0 y falta confirmarlo. La columna `fuente_sisam` conserva el origen.
- **Clima:** NASA POWER se descargó para los **543** municipios sin errores ni nulos, desde 2021-09-01. Con eso, `precip_acum_30d` y `precip_acum_120d` quedan completas desde la primera semana.
- **No se hizo en el preprocesamiento:** rezagos, targets, clases de riesgo, recorte de valores extremos, imputación, transformaciones ni particiones. Eso corresponde a las etapas siguientes.
